# Problem 3: Handle the API Failure

**Difficulty:** Medium | **Framework:** LangChain

**Categories:** error-recovery, tool-calling

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app/problems/handle-api-failure).

## 1. Install Dependencies

In [ ]:
!pip install -q langchain langchain-openai langchain-community langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The agent must retry failed API calls up to 3 times
- The agent must inform the user if all retries fail
- You may not change the simulated API behavior
- The final response must still be helpful even if some data is unavailable


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
import random
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

@tool
def fetch_stock_price(ticker: str) -> str:
    """Fetch the current stock price for a given ticker symbol."""
    # Simulated unreliable API - fails 60% of the time
    if random.random() < 0.6:
        raise ConnectionError(f"API timeout: Could not reach stock service for {ticker}")
    return f"{ticker}: $142.50 (+2.3%)"

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a financial assistant. Help users check stock prices."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# BUG: No error handling - agent crashes on API failure
agent = create_tool_calling_agent(llm, [fetch_stock_price], prompt)
executor = AgentExecutor(agent=agent, tools=[fetch_stock_price])

result = executor.invoke({"input": "What's the current price of AAPL?"})
print(result["output"])

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Wrap the tool call in error handling that catches exceptions and retries
2. Consider adding a fallback message when the API is completely unavailable
3. In LangGraph, you can create a retry loop as a conditional edge that checks for errors and routes back to the tool-calling node


## 7. Evaluation Criteria

Check your solution against these criteria:

- Implements retry logic
- Graceful failure message
- Succeeds when API recovers
- No unhandled exceptions
